# 🍺 Chopp & Cia · Inteligência de Risco em Comodato
# 📥 Notebook 01 — Ingestão e Consolidação

**Projeto Integrador VI** · 2º Semestre/2026 · FATEC Votorantim
**Alunos:** Ana Elisa · Arthur Nunes · Bruno Araujo · Caio Corrá · Lucas Camelo · Nicole Fava

---

## 🎯 Responsabilidade única deste notebook

Converter o backup relacional do ERP (`.sql`) em **uma tabela Delta versionada** no
Unity Catalog, com uma linha por cliente. Nada além disso: não há EDA aqui, não há
feature de modelo, não há treino.

| Entrada | Saída |
| :--- | :--- |
| `DB_POWER_SYS*.sql` (dump do ERP) | `<catálogo>.<schema>.dataset_consolidado_v<DATA_VERSION>` |

## 🔢 Por que a versão de dados existe

Cada execução deste notebook produz uma tabela **nova**, nomeada pela `DATA_VERSION`
que você declara no painel abaixo. As tabelas antigas continuam de pé.

Isso é o que torna um experimento reproduzível: um run do MLflow registra
`data_version='2.0'`, e essa tabela ainda existe amanhã, com exatamente as mesmas
linhas — mesmo que o ERP tenha sido recarregado três vezes desde então.

> **Regra:** mexeu no contrato de colunas, no parser ou nas regras de negócio deste
> notebook ⇒ **incremente `DATA_VERSION`** antes de rodar. Republicar sobre uma
> versão existente quebra a reprodutibilidade dos runs que apontam para ela — o
> notebook bloqueia isso por padrão.

## 🔒 Sem identificação nominal (nesta arquitetura)

`NM_PESSOA` e `DS_FANTASIA` **saíram do contrato de colunas**. Nenhum nome de cliente
é extraído do ERP em ponto algum do pipeline; a unidade de análise é `ID_PESSOA` do
começo ao fim.

A razão não é só de privacidade. Nome não tem poder preditivo sobre inadimplência — e
o que ele acrescentaria seria **viés**: um modelo com acesso a nomes pode aprender
clientes específicos em vez de comportamento de risco, e passa a discriminar por
identidade onde deveria discriminar por conduta. Numa aplicação de crédito, isso é
exatamente o erro que não se quer cometer.

> Para reidentificar um cliente a partir de `ID_PESSOA`, consulte o ERP — é onde o
> dado pessoal deve viver. O pipeline analítico não precisa dele.

## 🏗️ Etapas

| # | Etapa | Produz |
| :---: | :--- | :--- |
| 0 | Parametrização | caminhos e catálogo, sem nada fixo no código |
| 1 | Painel de controle | parâmetros da carga + `INGESTAO_HASH` |
| 2 | Contrato de colunas | `COLUNAS_NECESSARIAS` |
| 3 | Parser com projeção | `tabelas_db` (DataFrames por tabela) |
| 4 | Consolidação por cliente | `dataset` (1 linha por `ID_PESSOA`) |
| 5 | Auditoria | falha se o valor não se conservar |
| 6 | Publicação | tabela Delta + view corrente + catálogo de versões |

## 🎛️ Parâmetros (sem caminhos fixos)

Nenhum caminho de arquivo está escrito no código deste notebook. A origem dos
parâmetros depende de onde ele roda:

| Ambiente | Mecanismo | Onde aparece |
| :--- | :--- | :--- |
| **Databricks** | `dbutils.widgets` | campos no **topo** do notebook |
| **Local** (VS Code / Jupyter) | diálogo do sistema | janela de seleção de arquivo |


### Parâmetros deste notebook

| Parâmetro | O que é | Local | Databricks |
| :--- | :--- | :--- | :--- |
| `sql_file` | dump `.sql` do ERP | seletor de arquivo | caminho do Volume |
| `output_dir` | pasta dos CSVs de saída | seletor de pasta | *não usado* (publica no catálogo) |
| `catalogo` · `schema` | destino no Unity Catalog | *não usado* | campo de texto |
| `data_version` | versão desta carga | valor do painel | campo de texto |


### Como funciona localmente

Na primeira execução abre-se o diálogo do Windows. A escolha fica memorizada em
`~/.chopp_risco_params.json`, então **as execuções seguintes não perguntam nada** — o
diálogo só reaparece se o arquivo tiver sido movido, ou se você definir
`FORCAR_SELECAO = True`.

O diálogo é a **caixa nativa do Windows** (`comdlg32`), a mesma do Explorer — não o
`tkinter`. A diferença importa: o tkinter precisa criar uma janela-mãe para ancorar o
diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do editor.
A API nativa não cria janela nenhuma.

### 🔧 Se ainda assim o seletor não abrir

Em algumas instalações a janela simplesmente não aparece. Não é preciso lutar com
ela: preencha `CAMINHOS_MANUAIS` no topo da célula e o diálogo deixa de ser usado.

```python
CAMINHOS_MANUAIS = {
    "sql_file": r"C:\dados\DB_POWER_SYS.sql",
}
```

O `r` antes das aspas é necessário para que a barra invertida do Windows não seja
lida como caractere de escape. O que estiver preenchido ali tem **precedência sobre
tudo** — sobre o cache e sobre o diálogo.

### Como funciona no Databricks

Os campos aparecem no topo do notebook assim que a célula roda pela primeira vez.
Preencha e re-execute. Não há seletor de arquivo em notebook do Databricks — o caminho
do Volume é digitado, no formato `/Volumes/<catálogo>/<schema>/<volume>/arquivo`.



> **Sobre segurança:** tirar o caminho do código resolve **portabilidade** (o notebook
> roda na máquina de qualquer pessoa do grupo) e evita expor a estrutura de diretórios
> num repositório público. Não é um controle de acesso: quem executa o notebook lê o
> mesmo arquivo de qualquer forma. O controle de acesso real, no Databricks, vem das
> permissões do Unity Catalog sobre o Volume e as tabelas.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARAMETRIZAÇÃO SEM CAMINHOS FIXOS
#
#  Nenhum caminho de arquivo é escrito no código. A origem dos parâmetros
#  depende de onde o notebook roda:
#
#    Databricks → dbutils.widgets, os campos que aparecem no topo do notebook.
#                 É o mecanismo nativo da plataforma; não existe file picker
#                 em notebook do Databricks.
#    Local      → caixa de diálogo NATIVA do Windows (a mesma do Explorer),
#                 com a escolha memorizada num JSON. Sem tkinter: ele cria uma
#                 janela-mãe que, dentro do kernel do Jupyter, nasce sem foco e
#                 atrás do VS Code — o diálogo abria, mas ficava invisível.
#
#  O cache local existe para que a segunda execução não reabra o diálogo: ele
#  só volta a aparecer se o arquivo tiver sumido ou se você pedir explicitamente
#  com FORCAR_SELECAO = True.
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
from pathlib import Path

# Colocar em True reabre os diálogos mesmo havendo escolha memorizada.
# No Databricks não tem efeito: lá os widgets já são visíveis e editáveis.
FORCAR_SELECAO = True

# ── Caminhos informados à mão (alternativa ao diálogo) ────────────────────────
#
#  Se por qualquer motivo o seletor gráfico não abrir na sua máquina, preencha
#  aqui e o diálogo deixa de ser necessário. O que estiver preenchido TEM
#  PRECEDÊNCIA sobre tudo: sobre o cache e sobre o diálogo.
#
#  Deixe como está (strings vazias) para usar o seletor gráfico normalmente.
#
#  Use string "crua" (o r antes das aspas) para que a barra invertida do
#  Windows não seja interpretada como escape:
#      CAMINHOS_MANUAIS = {"sql_file": r"C:\dados\DB_POWER.sql"}
CAMINHOS_MANUAIS = {
    # "sql_file":         r"",
    # "output_dir":       r"",
    # "csv_consolidado":  r"",
    # "csv_split":        r"",
}

try:
    dbutils                                     # type: ignore # noqa: F821
    NO_DATABRICKS = True
except NameError:
    NO_DATABRICKS = False

# Onde a escolha local fica memorizada. Fica ao lado do notebook, e não no
# diretório de dados: é preferência de máquina, não dado do projeto.
_CACHE_PARAMS = Path.home() / ".chopp_risco_params.json"


def _cache_ler() -> dict:
    """Lê as escolhas memorizadas. Cache corrompido não pode derrubar o notebook."""
    try:
        if _CACHE_PARAMS.exists():
            return json.loads(_CACHE_PARAMS.read_text(encoding="utf-8"))
    except Exception:
        pass
    return {}


def _cache_gravar(chave: str, valor: str) -> None:
    """Memoriza uma escolha. Falha de escrita é irrelevante — só perde o atalho."""
    try:
        d = _cache_ler()
        d[chave] = str(valor)
        _CACHE_PARAMS.write_text(
            json.dumps(d, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════════
#  DIÁLOGO NATIVO DO WINDOWS (ctypes → comdlg32)
#
#  Por que não tkinter: o tkinter cria uma janela-mãe (Tk) para ancorar o
#  diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do
#  VS Code. O diálogo abre — mas fica invisível, e a célula parece travada.
#
#  A API abaixo é a mesma que o Explorer e qualquer programa Windows usam.
#  Não cria janela nenhuma: não há o que ficar atrás de nada.
# ══════════════════════════════════════════════════════════════════════════════
def _dialogo_windows_arquivo(titulo: str, filtros, inicial=None):
    """
    Caixa de seleção de arquivo nativa (GetOpenFileNameW). Caminho ou None.

    filtros: lista [(rótulo, padrão), ...] — ex.: [("Dump SQL", "*.sql")]
    """
    import ctypes
    from ctypes import wintypes

    class OPENFILENAMEW(ctypes.Structure):
        _fields_ = [
            ("lStructSize", wintypes.DWORD), ("hwndOwner", wintypes.HWND),
            ("hInstance", wintypes.HINSTANCE), ("lpstrFilter", wintypes.LPCWSTR),
            ("lpstrCustomFilter", wintypes.LPWSTR), ("nMaxCustFilter", wintypes.DWORD),
            ("nFilterIndex", wintypes.DWORD), ("lpstrFile", wintypes.LPWSTR),
            ("nMaxFile", wintypes.DWORD), ("lpstrFileTitle", wintypes.LPWSTR),
            ("nMaxFileTitle", wintypes.DWORD), ("lpstrInitialDir", wintypes.LPCWSTR),
            ("lpstrTitle", wintypes.LPCWSTR), ("Flags", wintypes.DWORD),
            ("nFileOffset", wintypes.WORD), ("nFileExtension", wintypes.WORD),
            ("lpstrDefExt", wintypes.LPCWSTR), ("lCustData", wintypes.LPARAM),
            ("lpfnHook", wintypes.LPVOID), ("lpTemplateName", wintypes.LPCWSTR),
            ("pvReserved", wintypes.LPVOID), ("dwReserved", wintypes.DWORD),
            ("FlagsEx", wintypes.DWORD),
        ]

    buf = ctypes.create_unicode_buffer(4096)

    # A API espera pares "rótulo\0padrão\0", terminados por um \0 extra.
    partes = []
    for rotulo, padrao in filtros:
        partes += [rotulo, padrao]
    filtro_api = "\0".join(partes) + "\0\0"

    ofn = OPENFILENAMEW()
    ofn.lStructSize = ctypes.sizeof(OPENFILENAMEW)
    ofn.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    ofn.lpstrFilter = filtro_api
    ofn.lpstrFile = ctypes.cast(buf, wintypes.LPWSTR)
    ofn.nMaxFile = 4096
    ofn.lpstrTitle = titulo
    ofn.lpstrInitialDir = inicial
    # NOCHANGEDIR: sem isto o diálogo muda o diretório de trabalho do kernel,
    # e caminhos relativos usados depois passam a apontar para outro lugar.
    ofn.Flags = 0x00001000 | 0x00000800 | 0x00000008 | 0x00080000

    if ctypes.windll.comdlg32.GetOpenFileNameW(ctypes.byref(ofn)):
        return buf.value or None
    return None


def _dialogo_windows_pasta(titulo: str):
    """Caixa de seleção de pasta nativa (SHBrowseForFolderW). Caminho ou None."""
    import ctypes
    from ctypes import wintypes

    class BROWSEINFOW(ctypes.Structure):
        _fields_ = [
            ("hwndOwner", wintypes.HWND), ("pidlRoot", ctypes.c_void_p),
            ("pszDisplayName", wintypes.LPWSTR), ("lpszTitle", wintypes.LPCWSTR),
            ("ulFlags", wintypes.UINT), ("lpfn", wintypes.LPVOID),
            ("lParam", wintypes.LPARAM), ("iImage", ctypes.c_int),
        ]

    shell32 = ctypes.windll.shell32
    nome = ctypes.create_unicode_buffer(4096)

    bi = BROWSEINFOW()
    bi.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    bi.pszDisplayName = ctypes.cast(nome, wintypes.LPWSTR)
    bi.lpszTitle = titulo
    bi.ulFlags = 0x00000001 | 0x00000040     # só diretórios + diálogo moderno

    shell32.SHBrowseForFolderW.restype = ctypes.c_void_p
    pidl = shell32.SHBrowseForFolderW(ctypes.byref(bi))
    if not pidl:
        return None
    try:
        caminho = ctypes.create_unicode_buffer(4096)
        shell32.SHGetPathFromIDListW.argtypes = [ctypes.c_void_p, wintypes.LPWSTR]
        ok = shell32.SHGetPathFromIDListW(pidl, caminho)
        return caminho.value if ok else None
    finally:
        # A lista de IDs é alocada pelo shell; liberá-la é responsabilidade nossa.
        ctypes.windll.ole32.CoTaskMemFree(ctypes.c_void_p(pidl))


def _dialogo_tkinter_arquivo(titulo: str, tipos, inicial=None):
    """Alternativa por tkinter, para quando a API do Windows não estiver disponível."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askopenfilename(
            parent=root, title=titulo, filetypes=tipos,
            initialdir=inicial or str(Path.home()),
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _dialogo_tkinter_pasta(titulo: str, inicial=None):
    """Alternativa por tkinter para seleção de pasta."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askdirectory(
            parent=root, title=titulo, initialdir=inicial or str(Path.home())
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _selecionar_arquivo(titulo: str, tipos, inicial=None):
    """
    Abre a caixa de seleção de arquivo. Retorna o caminho ou None.

    Tenta primeiro a API nativa do Windows (sem janela intermediária) e recorre
    ao tkinter fora do Windows ou se a API falhar.
    """
    if os.name == "nt":
        try:
            return _dialogo_windows_arquivo(titulo, tipos, inicial)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_arquivo(titulo, tipos, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def _selecionar_pasta(titulo: str, inicial=None):
    """Abre a caixa de seleção de pasta. Retorna o caminho ou None."""
    if os.name == "nt":
        try:
            return _dialogo_windows_pasta(titulo)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_pasta(titulo, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def widget(nome: str, padrao: str = "", rotulo: str = None,
           opcoes: list = None) -> str:
    """
    Declara um parâmetro de texto (ou dropdown) e devolve seu valor.

    No Databricks vira um campo no topo do notebook. Localmente devolve o valor
    memorizado, ou o padrão.

    A recriação do widget a cada execução é intencional: mudar o padrão no
    código passa a valer sem precisar remover o widget à mão.
    """
    rotulo = rotulo or nome
    if NO_DATABRICKS:
        try:
            if opcoes:
                dbutils.widgets.dropdown(nome, padrao or opcoes[0],
                                         opcoes, rotulo)          # noqa: F821
            else:
                dbutils.widgets.text(nome, padrao, rotulo)        # noqa: F821
        except Exception:
            pass   # widget já existe com outro tipo — o valor abaixo ainda serve
        try:
            return dbutils.widgets.get(nome)                      # noqa: F821
        except Exception:
            return padrao
    return _cache_ler().get(nome, padrao)


def parametro_arquivo(nome: str, titulo: str, tipos, padrao_databricks: str = "",
                      rotulo: str = None) -> str:
    """
    Resolve o caminho de um ARQUIVO de entrada.

    Databricks → widget de texto (caminho do Volume; não há file picker lá).
    Local      → escolha memorizada, ou diálogo do sistema.

    Falha com mensagem clara se nada for escolhido: seguir com caminho vazio
    produziria um FileNotFoundError muito adiante, sem contexto.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook\n"
                f"     com o caminho do arquivo no Volume, por exemplo:\n"
                f"       /Volumes/<catalogo>/<schema>/<volume>/arquivo.sql\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo — é a saída para quando o diálogo
    #    gráfico não abre.
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        if not os.path.exists(manual):
            raise FileNotFoundError(
                f"\n\n  ❌ CAMINHOS_MANUAIS['{nome}'] aponta para um arquivo que\n"
                f"     não existe:\n\n       {manual}\n\n"
                f"     Corrija o caminho no topo desta célula.\n"
            )
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada de uma execução anterior
    memorizado = _cache_ler().get(nome)
    if memorizado and os.path.exists(memorizado) and not FORCAR_SELECAO:
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        print(f"      (para escolher outro, defina FORCAR_SELECAO = True acima)")
        return memorizado

    if memorizado and not os.path.exists(memorizado):
        print(f"   ⚠️  O arquivo memorizado não existe mais:")
        print(f"      {memorizado}")

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de arquivo...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_arquivo(
        titulo, tipos,
        inicial=os.path.dirname(memorizado) if memorizado else None,
    )
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhum arquivo selecionado para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\o\\arquivo\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas\n"
            f"         (ícone do Python) antes de clicar em qualquer outro lugar.\n"
        )
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionado e memorizado para as próximas execuções.")
    return escolhido


def parametro_pasta(nome: str, titulo: str, padrao_databricks: str = "",
                    rotulo: str = None) -> str:
    """
    Resolve o caminho de um DIRETÓRIO de saída.

    Diferença em relação a parametro_arquivo(): um diretório inexistente é
    criado em vez de recusado — é saída, não entrada.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook.\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        os.makedirs(manual, exist_ok=True)
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada
    memorizado = _cache_ler().get(nome)
    if memorizado and not FORCAR_SELECAO:
        os.makedirs(memorizado, exist_ok=True)
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        return memorizado

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de pasta...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_pasta(titulo, inicial=memorizado)
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhuma pasta selecionada para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\a\\pasta\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas.\n"
        )
    os.makedirs(escolhido, exist_ok=True)
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionada e memorizada para as próximas execuções.")
    return escolhido


print("✅ Parametrização carregada")
print(f"   ├─ Ambiente : {'Databricks (widgets)' if NO_DATABRICKS else 'Local (seletor de arquivo)'}")
if not NO_DATABRICKS:
    print(f"   └─ Cache    : {_CACHE_PARAMS}")
else:
    print(f"   └─ Os parâmetros aparecem como campos no TOPO do notebook.")

# A seleção acontece nesta própria célula.
SQL_FILE = Path(parametro_arquivo(
    nome="sql_file",
    titulo="Selecione o dump SQL do ERP (.sql)",
    tipos=[("Dump SQL", "*.sql"), ("Todos os arquivos", "*.*")],
    padrao_databricks="/Volumes/projetointegrador/default/projetointegrador/DB_POWER_SYS.sql",
    rotulo="Arquivo .sql do dump (caminho no Volume)",
))

## 🎛️ 1. Painel de Controle da Ingestão

Toda variável capaz de mudar o **conteúdo da tabela publicada** mora nesta célula.
As seguintes apenas leem daqui.

| Bloco | O que controla | Efeito de mudar |
| :--- | :--- | :--- |
| `DATA_VERSION` | identidade da carga | cria tabela nova; runs antigos seguem válidos |
| `DESTINO` | catálogo · schema · prefixo | onde a tabela é publicada |
| `ORIGEM` | caminho do dump `.sql` | qual backup do ERP é lido |
| `REGRAS_NEGOCIO` | core business, aging, risco | **muda o dado** ⇒ exige nova versão |

O `INGESTAO_HASH`, calculado ao final, é o SHA-256 do contrato de colunas somado às
regras de negócio. Duas cargas com o mesmo hash aplicaram exatamente as mesmas
transformações — é o que permite provar, meses depois, que uma versão difere da
anterior porque o dump mudou, e não porque alguém editou uma regra pelo caminho.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PAINEL DE CONTROLE DA INGESTÃO
#  Tudo que muda o CONTEÚDO da tabela publicada está aqui. Nenhum número mágico
#  nas células seguintes: elas só leem deste bloco.
# ══════════════════════════════════════════════════════════════════════════════
import re
import csv
import os
import json
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Dict, Any

import pandas as pd
import numpy as np

# NO_DATABRICKS, widget(), parametro_arquivo() e parametro_pasta() vêm da
# célula de parametrização acima. Ela precisa ter sido executada.
if "parametro_arquivo" not in dir():
    raise NameError(
        "Execute a célula de PARAMETRIZAÇÃO (logo acima) antes desta. "
        "É ela que resolve os caminhos sem deixá-los fixos no código."
    )


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 · VERSÃO DOS DADOS  —  a decisão mais importante desta célula
#
#  Incremente ANTES de rodar sempre que mudar:
#    · o contrato de colunas (COLUNAS_NECESSARIAS)
#    · qualquer regra em REGRAS_NEGOCIO
#    · o arquivo de dump de origem
#
#  Convenção MAIOR.MENOR:
#    MAIOR → mudou o schema (coluna nova/removida) ou a semântica de uma coluna
#    MENOR → mesmo schema, dado novo (dump mais recente) ou correção de regra
# ══════════════════════════════════════════════════════════════════════════════
DATA_VERSION = widget("data_version", "1.0", "Versão dos dados (MAIOR.MENOR)")

DATA_VERSION_NOTA = (
    "Primeira carga da arquitetura modular (6 notebooks). Contrato de colunas "
    "sem identificacao nominal: nenhum nome de cliente sai do ERP em ponto "
    "nenhum do pipeline. Serve de ancora para comparar com as versoes seguintes."
)

# Proteção contra sobrescrita silenciosa. Publicar sobre uma versão existente faz
# os runs do MLflow que apontam para ela deixarem de ser reproduzíveis, sem que
# nada indique. Ligue conscientemente para recarregar a mesma versão.
PERMITIR_SOBRESCRITA = False


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 · DESTINO NO UNITY CATALOG
#  Catálogo e schema vêm de widget: variam entre workspaces e não devem estar
#  fixos no código.
# ══════════════════════════════════════════════════════════════════════════════
DESTINO = {
    "catalogo": widget("catalogo", "projetointegrador", "Catálogo (Unity Catalog)"),
    "schema": widget("schema", "projetointegrador", "Schema"),
    "prefixo_tabela": "dataset_consolidado",
}

# Nome final: dataset_consolidado_v1_0  (o ponto vira _, pois SQL não o aceita
# em identificador)
SUFIXO_VERSAO = DATA_VERSION.replace(".", "_")
TABELA_DESTINO = (
    f"{DESTINO['catalogo']}.{DESTINO['schema']}."
    f"{DESTINO['prefixo_tabela']}_v{SUFIXO_VERSAO}"
)

# View de conveniência apontando para a carga mais recente. Existe para quem quer
# "o dado atual" sem decorar sufixo; os notebooks de modelagem, porém, apontam
# para a TABELA versionada — uma view que muda debaixo de um experimento em curso
# é exatamente o que este desenho existe para evitar.
VIEW_CORRENTE = (
    f"{DESTINO['catalogo']}.{DESTINO['schema']}.{DESTINO['prefixo_tabela']}_corrente"
)

# Catálogo de versões: uma linha por carga publicada. É o índice que responde
# "quais versões existem e o que mudou entre elas" sem abrir seis notebooks.
TABELA_CATALOGO_VERSOES = (
    f"{DESTINO['catalogo']}.{DESTINO['schema']}.catalogo_versoes_dados"
)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 · ORIGEM DOS DADOS  —  resolvida SEM caminho fixo
#
#  Databricks → campo no topo do notebook (widget). Digite o caminho do Volume.
#  Local      → diálogo de seleção do Windows na primeira execução; depois a
#               escolha fica memorizada e o diálogo não reaparece.
#
#  A ausência de caminho no código é o que permite outra pessoa do grupo rodar
#  este notebook sem editar nada — e evita publicar a estrutura de diretórios da
#  sua máquina no repositório.
# ══════════════════════════════════════════════════════════════════════════════
# Saída local. No Databricks a publicação vai para o Unity Catalog e este
# diretório não é usado — por isso só é resolvido fora do Databricks.
if NO_DATABRICKS:
    OUTPUT_DIR = Path(".")
else:
    OUTPUT_DIR = Path(parametro_pasta(
        nome="output_dir",
        titulo="Selecione a pasta de saída dos CSVs",
    ))

ESCRITA_CSV = {"sep": ";", "encoding": "utf-8-sig", "index": False}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 · REGRAS DE NEGÓCIO
#  Mudar qualquer valor aqui muda o DADO publicado ⇒ exige nova DATA_VERSION.
#
#  ⚠️ Fronteira deliberada: estas regras definem o DADO, não o MODELO.
#     'limite_risco_eda' rotula PERFIL_RISCO para a EDA; o alvo ALTO_RISCO do
#     modelo é definido no notebook 03, a partir das TAXAS que ficam aqui. São
#     números distintos com o mesmo valor hoje — e devem poder divergir sem que
#     um arraste o outro.
# ══════════════════════════════════════════════════════════════════════════════
REGRAS_NEGOCIO: Dict[str, Any] = {

    # ── 4.1 Core business: o que é "chopp" para efeito de recorte ─────────────
    "core_business": {
        "regex_incluir": r"CHOPP|CHOPEIRA|BARRIL|CILINDRO|VÁLVULA|VALVULA|EXTRATORA|GÁS|GAS|CIL",
        "regex_excluir": r"COPO|DESCARTÁVEL|DESCARTAVEL|GELO|ÁGUA|AGUA|REFRIGERANTE|SUCO",
    },

    # ── 4.2 Aging: faixas de severidade do atraso ─────────────────────────────
    # Ordinal — a ordem É a escala. Alterar os cortes muda AGING_* de todos os
    # clientes. Formato: (limite_superior_em_dias, rótulo).
    "aging_faixas": [
        (0, "Sem Atraso"), (3, "1-3 Dias"), (7, "4-7 Dias"), (15, "8-15 Dias"),
        (20, "16-20 Dias"), (30, "21-30 Dias"), (float("inf"), "+30 Dias"),
    ],

    # ── 4.3 Rótulo de risco para a EDA ────────────────────────────────────────
    "limite_risco_eda": 0.20,

    # ── 4.4 Padronização de cidades ───────────────────────────────────────────
    # A fronteira seca com o Paraguai gera grafias múltiplas do mesmo município.
    # Sem agrupar, CIDADE vira feature de alta cardinalidade e baixo sinal.
    "mapa_cidades": [
        (["PONTA POR", "SANGA", "SANGRA"], "PONTA PORÃ"),
        (["PEDRO JUAN"], "PEDRO JUAN CABALLERO"),
        (["AMAMBA", "AMANBA"], "AMAMBAI"),
    ],
    "cidades_vazias": ["NAN", "", "NONE", "NÃO PREENCHIDO", "NAO PREENCHIDO"],
    "cidade_default": "OUTRAS CIDADES",

    # ── 4.5 Mapa de perfil da pessoa ──────────────────────────────────────────
    "mapa_perfil": {"F": "Física", "J": "Jurídica", "E": "Estrangeiro"},
}

# Ordem canônica das faixas de aging — usada como categoria ordenada adiante.
ORDEM_AGING = [rotulo for _, rotulo in REGRAS_NEGOCIO["aging_faixas"]]

print("=" * 78)
print(f"{'PAINEL DE CONTROLE DA INGESTÃO':^78}")
print("=" * 78)
print(f"  Versão dos dados : {DATA_VERSION}")
print(f"  Ambiente         : {'Databricks (Unity Catalog)' if NO_DATABRICKS else 'Local (CSV)'}")
print(f"  Origem           : {SQL_FILE.name}")
print(f"  Destino          : {TABELA_DESTINO if NO_DATABRICKS else OUTPUT_DIR}")
print(f"  Sobrescrita      : {'PERMITIDA' if PERMITIR_SOBRESCRITA else 'bloqueada'}")
print("-" * 78)
print(f"  Nota da versão   : {DATA_VERSION_NOTA[:60]}...")
print("=" * 78)

## 📋 2. Contrato de Colunas

O dump traz **133 tabelas** e centenas de colunas cada — `TB_PEDIDO_ITEM` sozinha
tem 200, quase todas fiscais (IBS, CBS, ICMS-ST, PIS, COFINS). Nenhuma entra num
modelo de risco de crédito.

O dicionário abaixo declara explicitamente cada campo consumido, com a justificativa
ao lado, e a projeção é aplicada **durante o parse** — as colunas fora do contrato
nunca chegam à memória.

> **Este dicionário é a interface com os notebooks seguintes.** Feature nova no
> notebook 03 ⇒ coluna de origem aqui, e `DATA_VERSION` incrementada.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONTRATO DE COLUNAS — o que os notebooks seguintes realmente consomem
#
#  Cada coluna abaixo foi rastreada até seu ponto de uso. Se uma coluna não está
#  aqui, é porque nenhuma célula de nenhum notebook a lê. Ao acrescentar uma
#  feature no notebook 03, acrescente a coluna de origem aqui e incremente
#  DATA_VERSION — este dicionário é o contrato de toda a cadeia.
# ══════════════════════════════════════════════════════════════════════════════
COLUNAS_NECESSARIAS: Dict[str, Dict[str, str]] = {

    # ── Dimensões cadastrais ──────────────────────────────────────────────────
    # ⚠️ NM_PESSOA e DS_FANTASIA foram REMOVIDAS do contrato no contrato de colunas.
    #    Motivo: são identificação nominal do cliente. Não entravam no modelo
    #    (eram barradas no notebook 03), mas circulavam por toda a cadeia sem
    #    necessidade — inclusive gravadas na tabela de treino.
    #    A unidade de análise é ID_PESSOA; nome não acrescenta poder preditivo
    #    e o que ele acrescentaria seria viés: um modelo que aprende nomes
    #    aprende clientes específicos, não comportamento de risco.
    #    Para reidentificar um cliente a partir de ID_PESSOA, consulte o ERP —
    #    é onde o dado pessoal deve viver.
    "TB_PESSOA": {
        "ID_PESSOA": "chave do cliente — unidade de análise de todo o projeto",
        "TP_TIPO": "F/J/E → feature categórica PERFIL",
    },
    "TB_CLIENTE": {
        "ID_CLIENTE": "chave usada pelo comodato",
        "ID_PESSOA": "ponte ID_CLIENTE ↔ ID_PESSOA (as trilhas usam chaves diferentes)",
        "ID_TIPO_ESTABELECIMENTO": "FK do segmento",
        "ID_FORMA_PAGAMENTO_1": "FK da forma de pagamento → feature PAGAMENTO",
    },
    "TB_CLIENTE_ENDERECO": {
        "ID_CLIENTE": "chave de junção",
        "DS_CIDADE": "→ feature categórica CIDADE (agrupada por agrupar_cidade)",
    },
    "TB_FORMA_PAGTO": {
        "ID_FORMA_PAGTO": "chave de junção",
        "DS_FORMA_PAGTO": "descrição → feature PAGAMENTO",
    },
    "TB_TIPO_ESTABELECIMENTO": {
        "ID_TIPO_ESTABELECIMENTO": "chave de junção",
        "DS_TIPO_ESTABELECIMENTO": "segmento do estabelecimento — EDA",
    },
    "TB_PRODUTO": {
        "ID_PRODUTO": "chave de junção",
        "DS_PRODUTO": "usado pelo regex de core business para isolar chopp/chopeira",
    },

    # ── Fato: itens de pedido (trilha de vendas) ──────────────────────────────
    "TB_PEDIDO_ITEM": {
        "ID_PEDIDO": "chave da transação",
        "ID_PRODUTO": "filtro core business + produto favorito",
        "QTD_VENDA": "volume consumido",
        "VL_FINANCEIRO": "receita do item → TOTAL_GASTO e TICKET_MEDIO",
    },
    # DT_PEDIDO e ID_PESSOA não vivem no item, e sim no cabeçalho do pedido.
    # Por isso TB_PEDIDO entra: é o que dá recência/frequência (RFM) ao modelo.
    "TB_PEDIDO": {
        "ID_PEDIDO": "chave de junção com o item",
        "ID_PESSOA": "dono do pedido",
        "DT_PEDIDO": "→ PRIMEIRA_COMPRA, ULTIMA_COMPRA, DIAS_DESDE_* e data_corte",
        "DT_ACERTO": "fechamento do pedido — ciclo de vida",
        "ID_STATUS": "situação do pedido",
    },

    # ── Fato: financeiro (trilha de inadimplência) ────────────────────────────
    # O dump NÃO tem 'TB_FINANCEIRO'. A informação de parcela vive em
    # TB_CONTAS_A_RECEBER_PARCELA e o vínculo com pedido/pessoa em
    # TB_CONTAS_A_RECEBER. Uma versão antiga deste pipeline procurava
    # TB_FINANCEIRO, não achava, e gravava um arquivo VAZIO — zerando
    # silenciosamente metade da variável-alvo.
    "TB_CONTAS_A_RECEBER": {
        "ID_CONTAS_A_RECEBER": "chave do título",
        "ID_PESSOA": "cliente devedor — chave da trilha financeira",
        "ID_PEDIDO": "pedido que originou a cobrança",
    },
    "TB_CONTAS_A_RECEBER_PARCELA": {
        "ID_CONTAS_A_RECEBER": "FK do título",
        "NR_PARCELA": "identifica o parcelamento",
        "DT_VENCIMENTO": "vencimento — base do cálculo de atraso",
        "DT_RECEBIMENTO": "data de recebimento (fallback de DT_BAIXA)",
        "DT_BAIXA": "liquidação efetiva — base do atraso financeiro",
        "VL_PARCELA": "valor da parcela",
        "VL_RECEBIDO": "detecta pagamento parcial",
        "TP_BAIXA": "auditoria de como o título foi liquidado",
    },

    # ── Fato: comodato (trilha de retenção de equipamento) ────────────────────
    "TB_COMODATO": {
        "ID_COMODATO": "chave do contrato → TOTAL_COMODATOS",
        "ID_CLIENTE": "⚠️ contém ID_PESSOA, não ID_CLIENTE — ver célula de consolidação",
        "ID_PEDIDO": "pedido vinculado",
        "DT_EMPRESTIMO": "início da cessão",
        "DT_VENCIMENTO": "prazo de devolução — base do atraso",
        "DT_RECOLHE": "devolução efetiva — base do atraso de comodato",
        "ID_STATUS": "situação do contrato",
    },
    "TB_COMODATO_BEM": {
        "ID_COMODATO": "FK do contrato",
        "ID_PRODUTO": "equipamento cedido — filtro core business",
        "QTD_PRODUTO": "unidades cedidas → QTD_EQUIPAMENTOS",
    },
}

TABELAS_ALVO = list(COLUNAS_NECESSARIAS)


# ══════════════════════════════════════════════════════════════════════════════
#  ASSINATURA DA INGESTÃO
#  Hash determinístico de (contrato + regras). Duas cargas com o mesmo hash
#  aplicaram as mesmas transformações: duas versões com hashes iguais diferem
#  pelo dump, não por uma regra editada pelo caminho.
# ══════════════════════════════════════════════════════════════════════════════
def calcular_hash(*blocos, tamanho: int = 12) -> str:
    """SHA-256 truncado de um JSON canônico (chaves ordenadas) dos blocos."""
    canonico = json.dumps(blocos, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(canonico.encode("utf-8")).hexdigest()[:tamanho]


INGESTAO_HASH = calcular_hash(COLUNAS_NECESSARIAS, REGRAS_NEGOCIO)

_total_cols = sum(len(v) for v in COLUNAS_NECESSARIAS.values())
print("📋 Contrato de colunas declarado")
print(f"   ├─ Tabelas          : {len(TABELAS_ALVO)}")
print(f"   ├─ Colunas          : {_total_cols}")
print(f"   └─ Hash da ingestão : {INGESTAO_HASH}")
print()
print("   💡 Feature nova no notebook 03 ⇒ coluna de origem aqui + nova DATA_VERSION.")

## 🔒 3. Guarda de Versão

Antes de gastar minutos parseando o dump, o notebook confere se `DATA_VERSION` já
foi publicada. Falhar agora custa segundos; descobrir depois que uma tabela referida
por doze runs do MLflow foi sobrescrita custa a rodada inteira de experimentos.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GUARDA DE VERSÃO — falha ANTES do trabalho pesado
#
#  Republicar sobre uma DATA_VERSION existente invalida, em silêncio, todo run
#  do MLflow que aponte para ela: o experimento diz 'data_version=2.0' e a
#  tabela 2.0 já não tem as linhas que produziram aquelas métricas.
# ══════════════════════════════════════════════════════════════════════════════
def tabela_existe(nome_completo: str) -> bool:
    """True se a tabela existe no Unity Catalog. Fora do Databricks, sempre False."""
    if not NO_DATABRICKS:
        return False
    try:
        return spark.catalog.tableExists(nome_completo)     # noqa: F821
    except Exception:
        return False


_ja_existe = tabela_existe(TABELA_DESTINO)

if _ja_existe and not PERMITIR_SOBRESCRITA:
    raise RuntimeError(
        f"\n\n  ❌ A tabela {TABELA_DESTINO} JÁ EXISTE.\n\n"
        f"     Publicar por cima invalidaria todo run do MLflow que aponta para\n"
        f"     data_version='{DATA_VERSION}' — as métricas ficariam sem os dados\n"
        f"     que as produziram.\n\n"
        f"     Escolha um caminho:\n"
        f"       (a) incremente DATA_VERSION no painel  ← o correto na maioria dos casos\n"
        f"       (b) PERMITIR_SOBRESCRITA = True, se esta carga é uma correção\n"
        f"           consciente de uma versão que ainda não foi usada em experimento\n"
    )

if _ja_existe:
    print(f"⚠️  {TABELA_DESTINO} já existe e será SOBRESCRITA (PERMITIR_SOBRESCRITA=True).")
    print(f"    Runs do MLflow com data_version='{DATA_VERSION}' deixarão de ser reproduzíveis.")
else:
    print(f"✅ Versão {DATA_VERSION} disponível para publicação.")

if not SQL_FILE.exists():
    raise FileNotFoundError(
        f"Dump não encontrado: {SQL_FILE}\n"
        f"   Ajuste ORIGEM['sql_file_{'volume' if NO_DATABRICKS else 'local'}'] no painel."
    )
print(f"✅ Dump localizado: {SQL_FILE.name} ({SQL_FILE.stat().st_size/1024/1024:.1f} MB)")

## 🛠️ 4. Parser com Projeção de Colunas

O backup traz comandos estruturais de servidores Windows incompatíveis com o ambiente
de análise. A rotina abaixo:

1. Varre o arquivo, tratando apenas `INSERT` das tabelas do contrato.
2. Resolve, na primeira ocorrência de cada tabela, o mapa **nome → posição** e guarda
   só os índices que interessam.
3. Neutraliza funções `CAST(...)` do SQL Server e alinha linhas encurtadas por `NULL`.
4. Materializa em memória **somente as colunas projetadas**.

> A projeção durante o parse não é otimização cosmética: em `TB_PEDIDO_ITEM`
> (24.399 linhas × 200 colunas) é a diferença entre carregar 4,9 milhões de células
> e carregar 98 mil.

### 🚨 Três falhas silenciosas que este parser detecta

| Sintoma | Causa | Consequência se passar |
| :--- | :--- | :--- |
| `CAST(N'2024-...' AS DateTime` numa célula | tipo parametrizado não coberto pelo regex | campos desalinhados, data inválida dezenas de células depois |
| `TP_TIPO` valendo `N'F'` | prefixo `N` de literal Unicode não removido | `PERFIL`/`CIDADE`/`PAGAMENTO` viram constante — 3 features mortas sem erro |
| Tabela do contrato ausente | dump parcial | coluna faltando no notebook 03, com mensagem muito menos clara |

Nenhuma delas lança exceção naturalmente. Por isso são checadas explicitamente.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARSER DO DUMP SQL — PROJEÇÃO DE COLUNAS NA LEITURA
#  As ~630 colunas fora do contrato são descartadas linha a linha e nunca chegam
#  à memória.
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ [1/3] Lendo o dump e projetando apenas as colunas do contrato...\n")

padrao_insert = re.compile(
    r"INSERT\s+\[dbo\]\.\[([A-Z0-9_]+)\]\s*\((.*?)\)\s*VALUES\s*\((.*)\);?",
    re.IGNORECASE,
)

# CAST(...) do SQL Server — o tipo pode vir parametrizado: Decimal(18, 2).
# `[A-Za-z]+(?:\([^)]*\))?` cobre tanto DateTime quanto Decimal(18, 2).
_RE_CAST_TEXTO = re.compile(
    r"CAST\(\s*N?'([^']*)'\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)
_RE_CAST_NUM = re.compile(
    r"CAST\(\s*([^'(),]*?)\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)


def _limpar(valor: str):
    """
    Normaliza um valor bruto vindo do dump.

    O SQL Server prefixa TODO literal de texto Unicode com N: N'CHOPP 50 LTS'.
    O csv.reader consome as aspas externas mas devolve o N e as internas, então o
    valor chega como o texto  N'CHOPP 50 LTS'  em vez de  CHOPP 50 LTS.

    Efeito prático se não tratado: TP_TIPO vira "N'F'" e o mapa {F, J, E} não
    casa — PERFIL fica "Não Informado" para 100% dos clientes. O mesmo vale para
    CIDADE e PAGAMENTO: 3 das 12 features do modelo silenciosamente degradadas a
    uma constante (o OneHotEncoder não reclama disso).
    """
    v = valor.strip()
    if v.upper() == "NULL":
        return None

    # Remove o prefixo N de literal Unicode, com ou sem a aspa de fechamento.
    # A aspa final pode faltar quando o próprio texto contém vírgula: o
    # csv.reader corta o campo ali e a sobra vai para o campo seguinte. Ex.:
    # N'CASAS NOTURNAS, BOATES'  →  campo 1: "N'CASAS NOTURNAS"
    # Preferimos o texto sem a aspa a manter o N' grudado no valor.
    if v[:2].upper() == "N'":
        v = v[2:]
    elif v.startswith("'"):
        v = v[1:]
    if v.endswith("'") and not v.endswith("''"):
        v = v[:-1]

    # '' é o escape de aspa simples dentro de string no SQL Server
    return v.replace("''", "'").strip()


def _registros_insert(caminho):
    """
    Gera um registro INSERT completo por iteração.

    Ler o arquivo linha a linha NÃO basta: campos de texto do ERP contêm quebras
    de linha literais (observações como 'CLIENTE NAO CONFIRMOU\n...'), e nesses
    casos um único INSERT ocupa várias linhas do dump. Cortar na quebra trunca o
    registro no meio — o efeito visível é um CAST sem parêntese de fechamento,
    que depois vira data inválida no pandas.

    Acumulamos linhas até o número de aspas simples ficar par (string fechada)
    E a linha terminar em ')' — só então o registro está completo.
    """
    buffer = ""
    with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
        for linha in f:
            if buffer:
                buffer += linha
            elif linha.startswith("INSERT"):
                buffer = linha
            else:
                continue

            if buffer.count("'") % 2 == 0 and buffer.rstrip().rstrip(";").endswith(")"):
                yield buffer
                buffer = ""
    if buffer:
        yield buffer


dados_memoria = {t: [] for t in TABELAS_ALVO}
colunas_arquivo = {}     # colunas como vêm no dump
indices_projecao = {}    # posições a manter, por tabela
linhas_descartadas = {t: 0 for t in TABELAS_ALVO}

for linha in _registros_insert(SQL_FILE):
    # A quebra de linha que estava DENTRO de um campo de texto vira espaço: o
    # valor continua legível e o registro passa a caber numa linha lógica, que é
    # o que o csv.reader espera.
    linha = linha.replace("\r", " ").replace("\n", " ")

    match = padrao_insert.search(linha)
    if not match:
        continue

    tabela = match.group(1).upper()
    if tabela not in COLUNAS_NECESSARIAS:
        continue

    # Na primeira ocorrência, resolve o mapa nome→posição desta tabela
    if tabela not in colunas_arquivo:
        cols_dump = [c.strip().strip("[]") for c in match.group(2).split(",")]
        colunas_arquivo[tabela] = cols_dump
        desejadas = COLUNAS_NECESSARIAS[tabela]
        indices_projecao[tabela] = [
            (cols_dump.index(c), c) for c in desejadas if c in cols_dump
        ]
        ausentes = [c for c in desejadas if c not in cols_dump]
        if ausentes:
            print(f"   ⚠️  {tabela}: colunas do contrato ausentes no dump → {ausentes}")

    # Neutraliza CAST(...) antes de passar ao csv.reader. Dois padrões, porque o
    # tipo pode ser PARAMETRIZADO:
    #   CAST(N'2024-04-25T13:51:55.320' AS DateTime)  → literal com aspas
    #   CAST(12.5 AS Decimal(18, 2))                  → literal numérico
    # O `(?:\([^)]*\))?` é o ponto crítico: sem ele, um `[^)]+` para no primeiro
    # `)` interno de Decimal(18, 2) e deixa um `)` órfão no meio da linha. O
    # csv.reader então desalinha TODOS os campos seguintes e a data chega ao
    # pandas como "CAST(N'...' AS DateTime" — corrompendo a linha inteira sem
    # lançar erro nenhum aqui.
    val_raw = _RE_CAST_TEXTO.sub(r"'\1'", match.group(3))
    val_raw = _RE_CAST_NUM.sub(r"\1", val_raw)

    leitor = csv.reader([val_raw], delimiter=",", quotechar="'", skipinitialspace=True)
    n_cols = len(colunas_arquivo[tabela])

    for row in leitor:
        valores = [_limpar(v) for v in row]

        # Alinhamento defensivo: o dump pode trazer linhas curtas por NULLs
        if len(valores) < n_cols:
            valores.extend([None] * (n_cols - len(valores)))
        elif len(valores) > n_cols:
            valores = valores[:n_cols]

        # ── PROJEÇÃO: só as posições do contrato entram na memória ────────────
        try:
            dados_memoria[tabela].append(
                [valores[i] for i, _ in indices_projecao[tabela]]
            )
        except IndexError:
            linhas_descartadas[tabela] += 1


# ── Materialização em DataFrames ─────────────────────────────────────────────
tabelas_db: Dict[str, pd.DataFrame] = {}
print(f"{'Tabela':<32} {'Linhas':>9} {'Cols dump':>10} {'Cols mantidas':>14}")
print("-" * 70)

total_orig = total_mantido = 0
for tabela in TABELAS_ALVO:
    if not dados_memoria[tabela]:
        print(f"{tabela:<32} {'—':>9}  (nenhum INSERT encontrado no dump)")
        continue

    nomes = [nome for _, nome in indices_projecao[tabela]]
    tabelas_db[tabela] = pd.DataFrame(dados_memoria[tabela], columns=nomes)

    n_dump = len(colunas_arquivo[tabela])
    total_orig += n_dump
    total_mantido += len(nomes)
    print(f"{tabela:<32} {len(tabelas_db[tabela]):>9,} {n_dump:>10} {len(nomes):>14}")

    if linhas_descartadas[tabela]:
        print(f"{'':<32} ⚠️  {linhas_descartadas[tabela]} linha(s) malformada(s) descartada(s)")

print("-" * 70)
print(f"{'TOTAL':<32} {'':>9} {total_orig:>10} {total_mantido:>14}")
if total_orig:
    print(f"\n   📉 Redução de colunas: {total_orig} → {total_mantido} "
          f"({100 - total_mantido / total_orig * 100:.1f}% descartado)")

# Libera a memória intermediária: com o DataFrame construído, as listas de listas
# são só cópia — e em TB_PEDIDO_ITEM elas não são pequenas.
del dados_memoria

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  AUDITORIA DO PARSE — as três falhas silenciosas
#  Nenhuma delas lança exceção sozinha; todas corrompem o dado adiante.
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. Tabela do contrato ausente ─────────────────────────────────────────────
# Falha cedo e alto: uma tabela faltando aqui vira coluna faltando no notebook 03,
# dezenas de células depois, com mensagem muito menos clara.
_faltando = [t for t in TABELAS_ALVO if t not in tabelas_db]
if _faltando:
    raise RuntimeError(
        f"Tabelas do contrato não encontradas no dump: {_faltando}. "
        f"Verifique se SQL_FILE aponta para o backup completo."
    )

# ── 2. Resíduo de CAST ────────────────────────────────────────────────────────
# Um CAST mal desfeito vira o texto "CAST(N'2024-04-25...' AS DateTime" dentro de
# uma célula, e só estoura muitas células depois, em pd.to_datetime, com mensagem
# que não aponta para a causa. Barato de checar, caro de diagnosticar depois.
_residuos = {}
for tabela, df in tabelas_db.items():
    n = sum(
        int(df[col].astype(str).str.contains("CAST(", regex=False, na=False).sum())
        for col in df.columns if df[col].dtype == "object"
    )
    if n:
        _residuos[tabela] = n

if _residuos:
    raise RuntimeError(
        f"Resíduo de CAST() encontrado após o parse: {_residuos}. "
        f"As linhas afetadas estão com os campos desalinhados. Verifique "
        f"_RE_CAST_TEXTO/_RE_CAST_NUM — provavelmente há um tipo parametrizado "
        f"novo no dump que os padrões não cobrem."
    )

# ── 3. Prefixo N' residual ────────────────────────────────────────────────────
# Não quebra nada aqui; só transforma uma feature categórica em constante lá na
# frente, onde o OneHotEncoder aceita sem reclamar.
_prefixo_n = {}
for tabela, df in tabelas_db.items():
    n = sum(
        int(df[col].astype(str).str.match(r"^N'").sum())
        for col in df.columns if df[col].dtype == "object"
    )
    if n:
        _prefixo_n[tabela] = n

if _prefixo_n:
    raise RuntimeError(
        f"Prefixo N' de literal Unicode não removido: {_prefixo_n}. Colunas de "
        f"texto ficariam com o valor errado (ex.: TP_TIPO=\"N'F'\" em vez de "
        f"\"F\"), degradando features categóricas a constantes. Verifique _limpar()."
    )

print("✅ Parse auditado:")
print("   ├─ todas as tabelas do contrato presentes")
print("   ├─ sem resíduo de CAST()")
print("   └─ sem prefixo N' de literal Unicode")

## 🔗 5. Consolidação por Cliente

As três trilhas transacionais são remontadas e consolidadas em **uma linha por
cliente** (`ID_PESSOA`).

| Trilha | Junção | Por que a junção é necessária |
| :--- | :--- | :--- |
| vendas | `TB_PEDIDO_ITEM` ⋈ `TB_PEDIDO` | o item não tem `ID_PESSOA` nem `DT_PEDIDO` — sem o cabeçalho não há RFM |
| financeiro | `TB_CONTAS_A_RECEBER_PARCELA` ⋈ `TB_CONTAS_A_RECEBER` | a parcela tem as datas; o título tem o cliente e o pedido |
| comodato | `TB_COMODATO` ⋈ `TB_COMODATO_BEM` | o contrato tem cliente e prazos; o bem diz qual equipamento e quantos |

### ⚠️ Por que não encadear as três num join só

Um pedido tem **N itens × M parcelas × K bens de comodato**. Encadeá-las por
`ID_PEDIDO` multiplica as linhas — medido neste dump: **24.399 → 43.562 (1,8×)** — e
faz o mesmo `VL_FINANCEIRO` ser contado várias vezes, **inflando o faturamento** sem
que nada indique o erro.

A consolidação correta **agrega cada trilha na sua própria granularidade** —
faturamento sobre itens, atrasos sobre parcelas, retenção sobre contratos — e só
então une os resultados pela chave do cliente. A auditoria confere que a soma do
faturamento por cliente bate **ao centavo** com a soma das linhas de origem.

### 🔑 A chave do comodato

`TB_COMODATO.ID_CLIENTE` tem nome de uma chave e **valores de outra**. Verificação
independente, usando o dono do pedido como fonte da verdade:

| Interpretação | Contratos coerentes com o dono do pedido |
| :--- | :--- |
| traduzir via `TB_CLIENTE` | 72 / 8.180 — **0,9%** |
| tratar o valor como `ID_PESSOA` | 8.180 / 8.180 — **100%** |

Como os dois espaços de ID se sobrepõem (`ID_CLIENTE` 5 ↔ `ID_PESSOA` 9, e existe uma
pessoa 5), a tradução não apenas perderia contratos: atribuiria inadimplência de
comodato **ao cliente errado**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MONTAGEM DAS TRILHAS TRANSACIONAIS
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔗 [2/3] Montando as trilhas transacionais...\n")


def _num(serie):
    """Converte para numérico preservando NaN — o dump entrega tudo como string."""
    return pd.to_numeric(serie, errors="coerce")


# ── TRILHA 1: vendas (item de pedido ⋈ cabeçalho) ─────────────────────────────
# O item não tem ID_PESSOA nem DT_PEDIDO: sem o cabeçalho não há RFM.
df_vendas = tabelas_db["TB_PEDIDO_ITEM"].merge(
    tabelas_db["TB_PEDIDO"][["ID_PEDIDO", "ID_PESSOA", "DT_PEDIDO", "DT_ACERTO", "ID_STATUS"]],
    on="ID_PEDIDO",
    how="inner",   # item órfão de pedido não tem dono nem data
)

# ── TRILHA 2: financeiro (parcela ⋈ título) ───────────────────────────────────
# A parcela tem as datas; o título tem ID_PESSOA e ID_PEDIDO.
df_financeiro = tabelas_db["TB_CONTAS_A_RECEBER_PARCELA"].merge(
    tabelas_db["TB_CONTAS_A_RECEBER"][["ID_CONTAS_A_RECEBER", "ID_PESSOA", "ID_PEDIDO"]],
    on="ID_CONTAS_A_RECEBER",
    how="inner",
)

# ── TRILHA 3: comodato (contrato ⋈ bem) ───────────────────────────────────────
df_comodato = tabelas_db["TB_COMODATO"].merge(
    tabelas_db["TB_COMODATO_BEM"][["ID_COMODATO", "ID_PRODUTO", "QTD_PRODUTO"]],
    on="ID_COMODATO",
    how="left",    # contrato sem bem registrado ainda é comodato em aberto
)

# ══════════════════════════════════════════════════════════════════════════════
#  ⚠️ CORREÇÃO DE CHAVE — TB_COMODATO.ID_CLIENTE contém ID_PESSOA
#
#  A coluna se chama ID_CLIENTE, mas os valores gravados nela são ID_PESSOA.
#  Verificação independente, usando o dono do pedido como fonte da verdade
#  (ID_PEDIDO do comodato → TB_PEDIDO.ID_PESSOA):
#
#     traduzir via TB_CLIENTE            →     72/8.180 batem (0,9%)
#     tratar o valor como ID_PESSOA      →  8.180/8.180 batem (100%)
#
#  Traduzir via TB_CLIENTE não apenas perdia contratos: dos que "resolvia",
#  quase todos apontavam para a PESSOA ERRADA — porque os IDs dos dois espaços
#  se sobrepõem. O efeito era inadimplência atribuída a quem não a tinha.
# ══════════════════════════════════════════════════════════════════════════════
df_comodato = df_comodato.rename(columns={"ID_CLIENTE": "ID_PESSOA"})

_dono_pedido = df_vendas[["ID_PEDIDO", "ID_PESSOA"]].drop_duplicates("ID_PEDIDO")
_chk = df_comodato.merge(
    _dono_pedido.rename(columns={"ID_PESSOA": "_DONO_PEDIDO"}),
    on="ID_PEDIDO", how="inner",
)
COERENCIA_CHAVE_COMODATO = float("nan")
if len(_chk):
    _coerentes = int((_num(_chk["ID_PESSOA"]) == _num(_chk["_DONO_PEDIDO"])).sum())
    COERENCIA_CHAVE_COMODATO = _coerentes / len(_chk) * 100
    print(f"   🔑 Chave do comodato: {_coerentes:,}/{len(_chk):,} contratos "
          f"({COERENCIA_CHAVE_COMODATO:.1f}%) coincidem com o dono do pedido.")
    if COERENCIA_CHAVE_COMODATO < 95:
        print("   ⚠️  Coerência abaixo do esperado — reveja a semântica de "
              "TB_COMODATO.ID_CLIENTE neste dump.")

print(f"\n   vendas     : {len(df_vendas):>7,} linhas (grão: item de pedido)")
print(f"   financeiro : {len(df_financeiro):>7,} linhas (grão: parcela)")
print(f"   comodato   : {len(df_comodato):>7,} linhas (grão: bem cedido)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  AGREGAÇÃO POR CLIENTE
#  Cada trilha é agregada na sua PRÓPRIA granularidade antes da união. É isso
#  que impede a duplicação: faturamento somado sobre itens, parcelas sobre
#  parcelas, comodatos sobre contratos — e só os RESULTADOS se encontram.
# ══════════════════════════════════════════════════════════════════════════════
print("\n🧮 [3/3] Consolidando por cliente...\n")

DATA_CORTE = pd.to_datetime(df_vendas["DT_PEDIDO"], errors="coerce").max()
print(f"   📅 Data de corte (último pedido): {DATA_CORTE:%d/%m/%Y}")


# ── TRILHA A: vendas → RFM ────────────────────────────────────────────────────
_v = df_vendas.copy()
_v["DT_PEDIDO"] = pd.to_datetime(_v["DT_PEDIDO"], errors="coerce")
_v["VL_FINANCEIRO"] = _num(_v["VL_FINANCEIRO"])
_v["QTD_VENDA"] = _num(_v["QTD_VENDA"])

agg_vendas = (
    _v.groupby("ID_PESSOA")
    .agg(
        PRIMEIRA_COMPRA=("DT_PEDIDO", "min"),
        ULTIMA_COMPRA=("DT_PEDIDO", "max"),
        FREQUENCIA_COMPRAS=("ID_PEDIDO", "nunique"),
        TOTAL_ITENS=("ID_PEDIDO", "count"),
        TOTAL_GASTO=("VL_FINANCEIRO", "sum"),
        QTD_TOTAL_VENDIDA=("QTD_VENDA", "sum"),
    )
    .reset_index()
)
agg_vendas["TICKET_MEDIO"] = agg_vendas["TOTAL_GASTO"] / agg_vendas["FREQUENCIA_COMPRAS"]
agg_vendas["DIAS_DESDE_PRIMEIRA_COMPRA"] = (DATA_CORTE - agg_vendas["PRIMEIRA_COMPRA"]).dt.days
agg_vendas["DIAS_DESDE_ULTIMA_COMPRA"] = (DATA_CORTE - agg_vendas["ULTIMA_COMPRA"]).dt.days

# Produto favorito: item mais consumido em volume. A EDA o exibe no ranking de
# clientes — sem ele seria preciso voltar ao grão transacional.
_favorito = (
    _v.groupby(["ID_PESSOA", "ID_PRODUTO"])["QTD_VENDA"].sum().reset_index()
    .sort_values(["ID_PESSOA", "QTD_VENDA"], ascending=[True, False])
    .drop_duplicates("ID_PESSOA")
    .merge(tabelas_db["TB_PRODUTO"][["ID_PRODUTO", "DS_PRODUTO"]], on="ID_PRODUTO", how="left")
    [["ID_PESSOA", "DS_PRODUTO"]]
    .rename(columns={"DS_PRODUTO": "PRODUTO_FAVORITO"})
)
agg_vendas = agg_vendas.merge(_favorito, on="ID_PESSOA", how="left")


# ── Marcação de core business ─────────────────────────────────────────────────
# Precisa ser feita AQUI, no grão transacional, porque depende de ID_PRODUTO —
# informação que não sobrevive à agregação por cliente. Vai como flag para a
# tabela final, e o notebook 03 filtra por ela.
_cfg_core = REGRAS_NEGOCIO["core_business"]
_prod_core = tabelas_db["TB_PRODUTO"].copy()
_prod_core["DS_PRODUTO"] = _prod_core["DS_PRODUTO"].fillna("").str.upper()
_ids_core = _prod_core[
    _prod_core["DS_PRODUTO"].str.contains(_cfg_core["regex_incluir"], regex=True)
    & ~_prod_core["DS_PRODUTO"].str.contains(_cfg_core["regex_excluir"], regex=True)
]["ID_PRODUTO"].unique()

_clientes_core = set(_v[_v["ID_PRODUTO"].isin(_ids_core)]["ID_PESSOA"].dropna())
_fat_core = _v[_v["ID_PRODUTO"].isin(_ids_core)]["VL_FINANCEIRO"].sum()

print(f"   🍺 Core business: {len(_ids_core)} produtos, {len(_clientes_core):,} clientes, "
      f"R$ {_fat_core:,.2f}")


# ── TRILHA B: financeiro → atraso de pagamento ────────────────────────────────
# Regra: atrasou se pagou depois do vencimento OU não pagou e o vencimento já
# passou. A segunda metade importa — sem ela o inadimplente que nunca pagou
# contaria como adimplente, por não ter data de baixa.
_f = df_financeiro.copy()
_f["DT_VENCIMENTO"] = pd.to_datetime(_f["DT_VENCIMENTO"], errors="coerce")
_col_pgto = "DT_BAIXA" if "DT_BAIXA" in _f.columns else "DT_RECEBIMENTO"
_f[_col_pgto] = pd.to_datetime(_f[_col_pgto], errors="coerce")
_f["VL_PARCELA"] = _num(_f["VL_PARCELA"])

_f["DIAS_ATRASO_PAG"] = (
    _f[_col_pgto].fillna(DATA_CORTE) - _f["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
_f["ATRASOU_PAGAMENTO"] = (
    (_f[_col_pgto] > _f["DT_VENCIMENTO"])
    | (_f[_col_pgto].isna() & (_f["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

agg_fin = (
    _f.groupby("ID_PESSOA")
    .agg(
        TOTAL_PARCELAS=("ID_CONTAS_A_RECEBER", "count"),
        PARCELAS_ATRASADAS=("ATRASOU_PAGAMENTO", "sum"),
        MEDIA_DIAS_ATRASO_PAG=("DIAS_ATRASO_PAG", "mean"),
        MAX_DIAS_ATRASO_PAG=("DIAS_ATRASO_PAG", "max"),
        VALOR_TOTAL_PARCELAS=("VL_PARCELA", "sum"),
    )
    .reset_index()
)
agg_fin["TAXA_ATRASO_PAGAMENTO"] = (
    agg_fin["PARCELAS_ATRASADAS"] / agg_fin["TOTAL_PARCELAS"].replace(0, 1)
)


# ── TRILHA C: comodato → atraso de devolução ──────────────────────────────────
# Atenção à granularidade: o contrato é a unidade de risco, mas a trilha tem uma
# linha por BEM. Deduplicamos por ID_COMODATO antes de contar contratos, senão um
# comodato com 3 chopeiras contaria como 3 atrasos.
_c = df_comodato.copy()
_c["DT_VENCIMENTO"] = pd.to_datetime(_c["DT_VENCIMENTO"], errors="coerce")
_c["DT_RECOLHE"] = pd.to_datetime(_c["DT_RECOLHE"], errors="coerce")
_c["QTD_PRODUTO"] = _num(_c["QTD_PRODUTO"])

_c["DIAS_ATRASO_COM"] = (
    _c["DT_RECOLHE"].fillna(DATA_CORTE) - _c["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
_c["ATRASOU_COMODATO"] = (
    (_c["DT_RECOLHE"] > _c["DT_VENCIMENTO"])
    | (_c["DT_RECOLHE"].isna() & (_c["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

# Unidades cedidas somam por bem; o risco conta por contrato.
_qtd_por_cliente = _c.groupby("ID_PESSOA")["QTD_PRODUTO"].sum().rename("QTD_EQUIPAMENTOS")
_contratos = _c.drop_duplicates("ID_COMODATO")

agg_com = (
    _contratos.groupby("ID_PESSOA")
    .agg(
        TOTAL_COMODATOS=("ID_COMODATO", "nunique"),
        COMODATOS_ATRASADOS=("ATRASOU_COMODATO", "sum"),
        MEDIA_DIAS_ATRASO_COM=("DIAS_ATRASO_COM", "mean"),
        MAX_DIAS_ATRASO_COM=("DIAS_ATRASO_COM", "max"),
    )
    .reset_index()
    .merge(_qtd_por_cliente, on="ID_PESSOA", how="left")
)
agg_com["TAXA_ATRASO_COMODATO"] = (
    agg_com["COMODATOS_ATRASADOS"] / agg_com["TOTAL_COMODATOS"].replace(0, 1)
)

print(f"   ✅ Agregações: vendas {len(agg_vendas):,} · financeiro {len(agg_fin):,} "
      f"· comodato {len(agg_com):,} clientes")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DIMENSÕES CADASTRAIS + UNIÃO FINAL
# ══════════════════════════════════════════════════════════════════════════════

def agrupar_cidade(cidade) -> str:
    """
    Padroniza variações de digitação segundo REGRAS_NEGOCIO['mapa_cidades'].

    A fronteira seca com o Paraguai produz grafias múltiplas do mesmo município
    ('PONTA PORA', 'PONTA PORÃ', 'PONTA PORAN'). Sem agrupar, CIDADE vira uma
    feature de altíssima cardinalidade e quase nenhum sinal.
    """
    c = str(cidade).upper().strip()
    for chaves, destino in REGRAS_NEGOCIO["mapa_cidades"]:
        if any(k in c for k in chaves):
            return destino
    if c in REGRAS_NEGOCIO["cidades_vazias"]:
        return "NÃO PREENCHIDO"
    return REGRAS_NEGOCIO["cidade_default"]


def categorizar_atraso(dias) -> str:
    """Converte dias de atraso na faixa de aging declarada em REGRAS_NEGOCIO."""
    d = pd.to_numeric(dias, errors="coerce")
    if pd.isna(d) or d <= 0:
        return REGRAS_NEGOCIO["aging_faixas"][0][1]
    for limite, rotulo in REGRAS_NEGOCIO["aging_faixas"]:
        if d <= limite:
            return rotulo
    return REGRAS_NEGOCIO["aging_faixas"][-1][1]


# ── Pessoa: perfil ────────────────────────────────────────────────────────────
# Sem NM_PESSOA : a identificação nominal saiu do contrato. O cliente é
# representado por ID_PESSOA em toda a cadeia.
dim = (
    tabelas_db["TB_PESSOA"][["ID_PESSOA", "TP_TIPO"]]
    .drop_duplicates("ID_PESSOA").copy()
)
dim["PERFIL"] = dim["TP_TIPO"].map(REGRAS_NEGOCIO["mapa_perfil"]).fillna("Não Informado")
dim = dim.drop(columns=["TP_TIPO"])

# ── Cliente: pagamento, segmento ──────────────────────────────────────────────
_cli = tabelas_db["TB_CLIENTE"][
    ["ID_CLIENTE", "ID_PESSOA", "ID_TIPO_ESTABELECIMENTO", "ID_FORMA_PAGAMENTO_1"]
].drop_duplicates("ID_PESSOA")

_cli = _cli.merge(
    tabelas_db["TB_FORMA_PAGTO"][["ID_FORMA_PAGTO", "DS_FORMA_PAGTO"]],
    left_on="ID_FORMA_PAGAMENTO_1", right_on="ID_FORMA_PAGTO", how="left",
).merge(
    tabelas_db["TB_TIPO_ESTABELECIMENTO"][["ID_TIPO_ESTABELECIMENTO", "DS_TIPO_ESTABELECIMENTO"]],
    on="ID_TIPO_ESTABELECIMENTO", how="left",
)
_cli["PAGAMENTO"] = _cli["DS_FORMA_PAGTO"].fillna("OUTROS").str.upper()
_cli["SEGMENTO"] = _cli["DS_TIPO_ESTABELECIMENTO"].fillna("OUTROS")

# ── Endereço: cidade padronizada ──────────────────────────────────────────────
_end = (
    tabelas_db["TB_CLIENTE_ENDERECO"][["ID_CLIENTE", "DS_CIDADE"]]
    .drop_duplicates("ID_CLIENTE").copy()
)
_end["CIDADE"] = _end["DS_CIDADE"].apply(agrupar_cidade)

dim_cliente = (
    _cli[["ID_CLIENTE", "ID_PESSOA", "PAGAMENTO", "SEGMENTO"]]
    .merge(_end[["ID_CLIENTE", "CIDADE"]], on="ID_CLIENTE", how="left")
    .drop_duplicates("ID_PESSOA")
)

# ── União: outer, para não perder cliente que só existe numa trilha ───────────
dataset = (
    dim.merge(dim_cliente.drop(columns=["ID_CLIENTE"]), on="ID_PESSOA", how="outer")
    .merge(agg_vendas, on="ID_PESSOA", how="outer")
    .merge(agg_fin, on="ID_PESSOA", how="outer")
    .merge(agg_com, on="ID_PESSOA", how="outer")
)

# Zero é a leitura correta aqui: "nenhuma parcela" são 0 parcelas, não um valor
# desconhecido. Datas ficam NaN de propósito — ver auditoria adiante.
_COLS_ZERO = [
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "TOTAL_GASTO", "QTD_TOTAL_VENDIDA", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "VALOR_TOTAL_PARCELAS", "TAXA_ATRASO_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM",
    "QTD_EQUIPAMENTOS", "TAXA_ATRASO_COMODATO",
]
dataset[_COLS_ZERO] = dataset[_COLS_ZERO].fillna(0)

for _col, _val in [("PERFIL", "Não Informado"), ("CIDADE", "NÃO PREENCHIDO"),
                   ("PAGAMENTO", "OUTROS"), ("SEGMENTO", "OUTROS")]:
    if _col in dataset.columns:
        dataset[_col] = dataset[_col].fillna(_val)

# ── Flags de cobertura ────────────────────────────────────────────────────────
# Baratas de gravar e evitam confundir "não tem comodato" com "comodato não
# encontrado" — leituras muito diferentes que um zero sozinho confunde.
dataset["CORE_BUSINESS"] = dataset["ID_PESSOA"].isin(_clientes_core).astype(int)
dataset["TEM_VENDAS"] = (dataset["FREQUENCIA_COMPRAS"] > 0).astype(int)
dataset["TEM_FINANCEIRO"] = (dataset["TOTAL_PARCELAS"] > 0).astype(int)
dataset["TEM_COMODATO"] = (dataset["TOTAL_COMODATOS"] > 0).astype(int)

# ── Colunas de apoio à EDA ────────────────────────────────────────────────────
# Aging pela severidade MÁXIMA, não pela média: um cliente com um atraso de 45
# dias é caso de "+30 Dias", ainda que a média o diluísse numa faixa branda.
dataset["AGING_PAGAMENTO"] = dataset["MAX_DIAS_ATRASO_PAG"].apply(categorizar_atraso)
dataset["AGING_COMODATO"] = dataset["MAX_DIAS_ATRASO_COM"].apply(categorizar_atraso)

dataset["MES_ULTIMA_COMPRA"] = (
    pd.to_datetime(dataset["ULTIMA_COMPRA"], errors="coerce").dt.to_period("M").astype(str)
)

_LIM = REGRAS_NEGOCIO["limite_risco_eda"]
dataset["RISCO_FINANCEIRO"] = (dataset["TAXA_ATRASO_PAGAMENTO"] > _LIM).astype(int)
dataset["RISCO_COMODATO"] = (dataset["TAXA_ATRASO_COMODATO"] > _LIM).astype(int)
dataset["PERFIL_RISCO"] = np.select(
    [
        (dataset["RISCO_FINANCEIRO"] == 1) & (dataset["RISCO_COMODATO"] == 1),
        dataset["RISCO_FINANCEIRO"] == 1,
        dataset["RISCO_COMODATO"] == 1,
    ],
    ["RISCO DUPLO", "SÓ FINANCEIRO", "SÓ COMODATO"],
    default="SEM RISCO",
)

# ── Metadados de proveniência: viajam COM o dado ──────────────────────────────
# É o que permite, olhando só a tabela, saber qual carga a produziu — sem
# depender de o nome da tabela ter sido preservado numa cópia.
dataset["_DATA_VERSION"] = DATA_VERSION
dataset["_INGESTAO_HASH"] = INGESTAO_HASH
dataset["_CARGA_TS"] = pd.Timestamp.now()

_ORDEM = [
    "ID_PESSOA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO",
    "PRIMEIRA_COMPRA", "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "PRODUTO_FAVORITO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS", "AGING_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS", "AGING_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
    "_DATA_VERSION", "_INGESTAO_HASH", "_CARGA_TS",
]
dataset = dataset[[c for c in _ORDEM if c in dataset.columns]]

print(f"✅ dataset consolidado: {dataset.shape[0]:,} clientes × {dataset.shape[1]} colunas")

## 🔍 6. Auditoria de Consolidação

Três verificações que **falham a carga** em vez de deixar o erro seguir adiante:

1. **Conservação de valor** — a soma de `TOTAL_GASTO` por cliente tem de bater ao
   centavo com a soma das linhas de origem. É a checagem que detecta a inflação de
   um join em cadeia.
2. **Chave única** — `ID_PESSOA` sem duplicatas: a promessa de "1 linha por cliente".
3. **Features categóricas vivas** — `PERFIL`/`CIDADE`/`PAGAMENTO` com mais de um
   valor. Uma categórica constante é o sintoma do bug do prefixo `N'`, e treina sem
   erro nenhum.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  AUDITORIA — falha a carga em vez de propagar o erro
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 78)
print(f"{'AUDITORIA DE CONSOLIDAÇÃO':^78}")
print("=" * 78)

# ── 1. Conservação de valor ───────────────────────────────────────────────────
_fat_origem = float(_v["VL_FINANCEIRO"].sum())
_fat_consol = float(dataset["TOTAL_GASTO"].sum())
_dif = abs(_fat_origem - _fat_consol)

print(f"\n  Faturamento nas vendas (origem) : R$ {_fat_origem:>15,.2f}")
print(f"  Faturamento no consolidado      : R$ {_fat_consol:>15,.2f}")
print(f"  Diferença                       : R$ {_dif:>15,.2f}")
if _dif > 0.01:
    raise RuntimeError(
        f"Faturamento não conservado na consolidação (diferença R$ {_dif:,.2f}). "
        f"Provável duplicação de linhas em algum merge — verifique se alguma "
        f"trilha foi unida antes de ser agregada."
    )
print("  ✅ Valor conservado — nenhuma duplicação nos merges.")

# ── 2. Chave única ────────────────────────────────────────────────────────────
if dataset["ID_PESSOA"].duplicated().any():
    _dups = int(dataset["ID_PESSOA"].duplicated().sum())
    raise RuntimeError(
        f"{_dups} ID_PESSOA duplicado(s) — a tabela deveria ter 1 linha por cliente."
    )
print(f"  ✅ Chave única: {len(dataset):,} clientes, um por linha.")

# ── 3. Categóricas vivas ──────────────────────────────────────────────────────
_constantes = [c for c in ["PERFIL", "CIDADE", "PAGAMENTO"] if dataset[c].nunique() <= 1]
if _constantes:
    raise RuntimeError(
        f"Features categóricas constantes: {_constantes}. É o sintoma do prefixo "
        f"N' não removido — a feature treina sem erro e não informa nada. "
        f"Verifique _limpar()."
    )
print(f"  ✅ Categóricas com variação: PERFIL {dataset['PERFIL'].nunique()} · "
      f"CIDADE {dataset['CIDADE'].nunique()} · PAGAMENTO {dataset['PAGAMENTO'].nunique()}")

# ── Cobertura (informativo) ───────────────────────────────────────────────────
print(f"\n  Cobertura por trilha:")
for _flag, _rot in [("TEM_VENDAS", "com vendas"), ("TEM_FINANCEIRO", "com financeiro"),
                    ("TEM_COMODATO", "com comodato"), ("CORE_BUSINESS", "core business")]:
    _n = int(dataset[_flag].sum())
    print(f"    ├─ {_rot:<16}: {_n:>5,} clientes ({_n/len(dataset)*100:>5.1f}%)")

# Datas nulas são semanticamente corretas: cliente sem venda não tem primeira nem
# última compra. Preencher com 0 mentiria ("comprou hoje") e com -1 inventaria uma
# data. Ficam NaN — mas a contagem é reportada para não virar surpresa adiante.
_sem_venda = int((dataset["TEM_VENDAS"] == 0).sum())
if _sem_venda:
    print(f"\n  ℹ️  {_sem_venda} cliente(s) sem venda têm PRIMEIRA_COMPRA, ULTIMA_COMPRA")
    print(f"      e DIAS_DESDE_* nulos — é a leitura correta, não um defeito.")

# ── Estatísticas da carga, para o catálogo de versões ────────────────────────
ESTATISTICAS_CARGA = {
    "n_clientes": int(len(dataset)),
    "n_colunas": int(dataset.shape[1]),
    "n_core_business": int(dataset["CORE_BUSINESS"].sum()),
    "n_com_vendas": int(dataset["TEM_VENDAS"].sum()),
    "n_com_financeiro": int(dataset["TEM_FINANCEIRO"].sum()),
    "n_com_comodato": int(dataset["TEM_COMODATO"].sum()),
    "faturamento_total": round(_fat_consol, 2),
    "data_corte": str(DATA_CORTE.date()),
    "coerencia_chave_comodato_pct": round(float(COERENCIA_CHAVE_COMODATO), 2),
    "taxa_atraso_financeiro_pct": round(
        float(dataset["PARCELAS_ATRASADAS"].sum() / max(dataset["TOTAL_PARCELAS"].sum(), 1) * 100), 2),
    "taxa_atraso_comodato_pct": round(
        float(dataset["COMODATOS_ATRASADOS"].sum() / max(dataset["TOTAL_COMODATOS"].sum(), 1) * 100), 2),
}

print("\n" + "=" * 78)
print("  ✅ AUDITORIA APROVADA — dataset pronto para publicação")
print("=" * 78)

## 💾 7. Publicação no Unity Catalog

A escrita usa `saveAsTable()`, e não `save(caminho)`. A distinção não é cosmética:

| | `.save("/Volumes/...")` | `.saveAsTable("cat.schema.tabela")` |
| :--- | :--- | :--- |
| **O que cria** | arquivos Delta soltos num Volume | **tabela gerenciada** no catálogo |
| **Aparece em** | listagem de arquivos | Catalog Explorer, `SHOW TABLES`, `information_schema` |
| **Consulta** | só por caminho | `SELECT * FROM cat.schema.tabela` |
| **Governança** | permissão do Volume | GRANT por tabela, lineage, comentários |

Era a origem dos `.parquet` que você via: `save()` num Volume grava os data files do
Delta, e o que fica visível ao navegar são os `part-*.snappy.parquet` — o formato de
arquivo *dentro* do Delta.

Além da tabela, esta célula publica:

- **Comentários de coluna** — o dicionário de dados vive no catálogo, não num `.docx`
- **`TBLPROPERTIES`** — `data_version`, `ingestao_hash`, nota da carga, consultáveis por SQL
- **View `_corrente`** — aponta para a última versão publicada, para consulta ad-hoc
- **`catalogo_versoes_dados`** — uma linha por carga, o índice de todas as versões

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PUBLICAÇÃO — TABELA GERENCIADA NO UNITY CATALOG
#
#  saveAsTable(), não save(). A diferença:
#    .save("/Volumes/...")        → arquivos Delta soltos; o que se vê ao navegar
#                                   são os part-*.snappy.parquet de dentro do Delta
#    .saveAsTable("cat.sch.tab")  → TABELA no catálogo: SHOW TABLES, SELECT,
#                                   GRANT, comentários, lineage
# ══════════════════════════════════════════════════════════════════════════════

def _preparar_para_spark(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza um DataFrame pandas para escrita via Spark.

    Duas correções necessárias e não óbvias:
      1. Spark não aceita nome de coluna com espaço ou pontuação;
      2. coluna 'object' contendo só NaN vira NullType no Spark e quebra a
         escrita — as de texto são forçadas a string.
    """
    out = df.copy()
    out.columns = [
        str(c).strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for c in out.columns
    ]
    for col in out.columns:
        if out[col].dtype == "object":
            # NaN vira o texto 'nan' com astype(str) — preenchemos antes para
            # que a ausência continue legível como ausência.
            out[col] = out[col].fillna("").astype(str)
    return out


# Dicionário de dados: vira COMMENT de coluna no catálogo. Documentação que fica
# onde o dado está, e não num anexo que se perde.
COMENTARIOS_COLUNAS = {
    "ID_PESSOA": "Chave do cliente (unidade de análise de todo o projeto)",
    "PERFIL": "Física | Jurídica | Estrangeiro (de TB_PESSOA.TP_TIPO)",
    "CIDADE": "Município padronizado (agrupa variações de digitação)",
    "PAGAMENTO": "Forma de pagamento padrão do cadastro",
    "SEGMENTO": "Tipo de estabelecimento (cadastro; sabidamente ruidoso)",
    "PRIMEIRA_COMPRA": "Data do primeiro pedido; nulo se o cliente nunca comprou",
    "ULTIMA_COMPRA": "Data do último pedido; nulo se o cliente nunca comprou",
    "MES_ULTIMA_COMPRA": "Período AAAA-MM da última compra (recência)",
    "DIAS_DESDE_PRIMEIRA_COMPRA": "Dias entre a primeira compra e a data de corte",
    "DIAS_DESDE_ULTIMA_COMPRA": "Dias entre a última compra e a data de corte (R do RFM)",
    "FREQUENCIA_COMPRAS": "Pedidos distintos do cliente (F do RFM)",
    "TOTAL_ITENS": "Linhas de item de pedido",
    "QTD_TOTAL_VENDIDA": "Unidades vendidas somadas",
    "TOTAL_GASTO": "Receita total do cliente (M do RFM)",
    "TICKET_MEDIO": "TOTAL_GASTO / FREQUENCIA_COMPRAS",
    "PRODUTO_FAVORITO": "Produto mais consumido em volume",
    "TOTAL_PARCELAS": "Parcelas de contas a receber do cliente",
    "PARCELAS_ATRASADAS": "Parcelas pagas em atraso ou vencidas e não pagas",
    "TAXA_ATRASO_PAGAMENTO": "PARCELAS_ATRASADAS / TOTAL_PARCELAS — base do alvo",
    "MEDIA_DIAS_ATRASO_PAG": "Média de dias de atraso no pagamento",
    "MAX_DIAS_ATRASO_PAG": "Pior atraso de pagamento observado",
    "VALOR_TOTAL_PARCELAS": "Soma do valor das parcelas",
    "AGING_PAGAMENTO": "Faixa de severidade do PIOR atraso de pagamento",
    "TOTAL_COMODATOS": "Contratos de comodato do cliente",
    "COMODATOS_ATRASADOS": "Contratos devolvidos em atraso ou vencidos e não devolvidos",
    "TAXA_ATRASO_COMODATO": "COMODATOS_ATRASADOS / TOTAL_COMODATOS — base do alvo",
    "MEDIA_DIAS_ATRASO_COM": "Média de dias de atraso na devolução",
    "MAX_DIAS_ATRASO_COM": "Pior atraso de devolução observado",
    "QTD_EQUIPAMENTOS": "Unidades de equipamento cedidas",
    "AGING_COMODATO": "Faixa de severidade do PIOR atraso de devolução",
    "RISCO_FINANCEIRO": f"1 se TAXA_ATRASO_PAGAMENTO > {REGRAS_NEGOCIO['limite_risco_eda']} (rótulo de EDA)",
    "RISCO_COMODATO": f"1 se TAXA_ATRASO_COMODATO > {REGRAS_NEGOCIO['limite_risco_eda']} (rótulo de EDA)",
    "PERFIL_RISCO": "RISCO DUPLO | SÓ FINANCEIRO | SÓ COMODATO | SEM RISCO",
    "CORE_BUSINESS": "1 se o cliente comprou chopp/chopeira — universo de modelagem",
    "TEM_VENDAS": "1 se o cliente tem histórico de vendas",
    "TEM_FINANCEIRO": "1 se o cliente tem parcelas registradas",
    "TEM_COMODATO": "1 se o cliente tem contratos de comodato",
    "_DATA_VERSION": "Versão desta carga — referenciada pelos runs do MLflow",
    "_INGESTAO_HASH": "Hash do contrato de colunas + regras que produziram a carga",
    "_CARGA_TS": "Momento da execução desta ingestão",
}


if NO_DATABRICKS:
    print("=" * 78)
    print(f"{'PUBLICAÇÃO NO UNITY CATALOG':^78}")
    print("=" * 78)
    print(f"  Tabela : {TABELA_DESTINO}\n")

    sdf = spark.createDataFrame(_preparar_para_spark(dataset))   # noqa: F821

    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABELA_DESTINO)
    )
    print(f"   ✅ Tabela gravada: {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas")

    # ── Propriedades: metadados consultáveis por SQL ──────────────────────────
    # DESCRIBE DETAIL / SHOW TBLPROPERTIES devolvem isto sem abrir o notebook.
    _nota_sql = DATA_VERSION_NOTA.replace("'", "''")
    spark.sql(f"""
        ALTER TABLE {TABELA_DESTINO} SET TBLPROPERTIES (
            'data_version'   = '{DATA_VERSION}',
            'ingestao_hash'  = '{INGESTAO_HASH}',
            'origem_dump'    = '{SQL_FILE.name}',
            'carga_ts'       = '{datetime.now().isoformat()}',
            'nota'           = '{_nota_sql}',
            'n_clientes'     = '{ESTATISTICAS_CARGA["n_clientes"]}',
            'n_core_business'= '{ESTATISTICAS_CARGA["n_core_business"]}'
        )
    """)                                                             # noqa: F821

    spark.sql(f"""
        COMMENT ON TABLE {TABELA_DESTINO} IS
        'Dataset consolidado de risco Chopp & Cia — 1 linha por cliente (ID_PESSOA).
         Versão {DATA_VERSION} · hash {INGESTAO_HASH} · origem {SQL_FILE.name}.
         Produzido por Chopp_Risco_01_Ingestao. Consumido por 02_EDA e 03_Preparacao.'
    """)                                                             # noqa: F821
    print("   ✅ Propriedades e comentário da tabela aplicados")

    # ── Comentários de coluna: o dicionário de dados no catálogo ──────────────
    _n_com = 0
    for _col, _txt in COMENTARIOS_COLUNAS.items():
        if _col in dataset.columns:
            try:
                spark.sql(                                           # noqa: F821
                    f"ALTER TABLE {TABELA_DESTINO} ALTER COLUMN {_col} "
                    f"COMMENT '{_txt.replace(chr(39), chr(39)*2)}'"
                )
                _n_com += 1
            except Exception as e:
                print(f"      ⚠️  comentário de {_col} não aplicado: {type(e).__name__}")
    print(f"   ✅ {_n_com} comentários de coluna aplicados (dicionário de dados)")

    # ── View corrente ─────────────────────────────────────────────────────────
    # Conveniência para consulta ad-hoc. Os notebooks de modelagem NÃO leem daqui:
    # uma view que muda debaixo de um experimento em curso é exatamente o que
    # este desenho versionado existe para evitar.
    spark.sql(f"""
        CREATE OR REPLACE VIEW {VIEW_CORRENTE}
        COMMENT 'Aponta para a carga mais recente ({TABELA_DESTINO}). Para
                 experimentos, use a TABELA versionada — não esta view.'
        AS SELECT * FROM {TABELA_DESTINO}
    """)                                                             # noqa: F821
    print(f"   ✅ View corrente atualizada: {VIEW_CORRENTE}")

else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    _path_csv = OUTPUT_DIR / f"dataset_consolidado_v{SUFIXO_VERSAO}.csv"
    dataset.to_csv(_path_csv, **ESCRITA_CSV)
    print("=" * 78)
    print(f"{'PUBLICAÇÃO LOCAL (fora do Databricks)':^78}")
    print("=" * 78)
    print(f"   ✅ {_path_csv.name}  ({_path_csv.stat().st_size/1024/1024:.2f} MB)")
    print(f"      {dataset.shape[0]:,} clientes × {dataset.shape[1]} colunas")
    print(f"\n   ℹ️  Para publicar no Unity Catalog, execute este notebook no Databricks.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CATÁLOGO DE VERSÕES — o índice de todas as cargas
#  Responde "quais versões existem e o que mudou entre elas" com um SELECT,
#  sem abrir seis notebooks nem depender da memória de ninguém.
# ══════════════════════════════════════════════════════════════════════════════
registro_versao = {
    "data_version": DATA_VERSION,
    "tabela": TABELA_DESTINO,
    "ingestao_hash": INGESTAO_HASH,
    "origem_dump": SQL_FILE.name,
    "carga_ts": datetime.now().isoformat(),
    "nota": DATA_VERSION_NOTA,
    **{k: str(v) for k, v in ESTATISTICAS_CARGA.items()},
}

if NO_DATABRICKS:
    _sdf_reg = spark.createDataFrame(pd.DataFrame([registro_versao]))   # noqa: F821

    # append: o catálogo é um log de cargas, não um snapshot. Uma versão
    # republicada aparece duas vezes — e é isso que se quer ver.
    (
        _sdf_reg.write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(TABELA_CATALOGO_VERSOES)
    )
    print(f"📚 Versão registrada em {TABELA_CATALOGO_VERSOES}\n")

    _hist = spark.sql(f"""
        SELECT data_version, tabela, ingestao_hash, n_clientes,
               n_core_business, carga_ts
        FROM {TABELA_CATALOGO_VERSOES}
        ORDER BY carga_ts DESC
    """).toPandas()                                                     # noqa: F821
    print("   Versões publicadas até agora:")
    print(_hist.to_string(index=False))
else:
    _path_reg = OUTPUT_DIR / "catalogo_versoes_dados.csv"
    _df_reg = pd.DataFrame([registro_versao])
    if _path_reg.exists():
        _df_reg = pd.concat([pd.read_csv(_path_reg, sep=";"), _df_reg], ignore_index=True)
    _df_reg.to_csv(_path_reg, **ESCRITA_CSV)
    print(f"📚 Versão registrada em {_path_reg.name}")

print("\n" + "=" * 78)
print(f"{'🎉 INGESTÃO CONCLUÍDA':^78}")
print("=" * 78)
print(f"  Versão publicada : {DATA_VERSION}")
print(f"  Tabela           : {TABELA_DESTINO if NO_DATABRICKS else 'CSV local'}")
print(f"  Hash da ingestão : {INGESTAO_HASH}")
print(f"  Clientes         : {ESTATISTICAS_CARGA['n_clientes']:,} "
      f"({ESTATISTICAS_CARGA['n_core_business']:,} no core business)")
print("-" * 78)
print("  PRÓXIMO PASSO — nos notebooks 02 e 03, aponte o parâmetro de entrada para:")
print(f"     TABELA_ENTRADA = \"{TABELA_DESTINO}\"")
print()
print("  Esse nome é o que amarra um experimento do MLflow aos dados exatos que o")
print("  produziram. Não use a view _corrente em experimento: ela muda de baixo.")
print("=" * 78)